# 08 - Ensemble dos 3 Modelos


In [6]:
import os
import sys
import pickle
import random
import json

sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from torch.utils.data import DataLoader

from src.data_processing import clean_text, clean_text_sequences
from src.features import TextDataset, Vocabulary, load_glove_embeddings, texts_to_sequences
from src.models_numpy.dnn import CategoricalCrossEntropy, Dataset, NeuralNetwork, accuracy
from src.models_pytorch.cnn1d import CNN1DClassifier
from src.models_pytorch.distilbert import DistilBERTClassifier, DistilBERTDataset, get_tokenizer


In [7]:
ROOT = os.path.abspath('..')
SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASSES = ['Anthropic', 'Google', 'Human', 'Meta', 'OpenAI']
LABEL_MAP = {label: i for i, label in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

TRAIN_PATH = os.path.join(ROOT, 'data', 'processed', 'dataset_combined.csv')
EXEMPLOS_PATH = os.path.join(ROOT, 'data', 'validation', 'dataset-exemplos.csv')
EARLY_STOP_PATH = os.path.join(ROOT, 'data', 'validation', 'subm1_labels_revealed.csv')
EVAL_PATH = os.path.join(ROOT, 'data', 'validation', 'subm2_labels_revealed.csv')
GLOVE_PATH = os.path.join(ROOT, 'data', 'embeddings', 'glove.6B.100d.txt')
SAVED_MODELS_DIR = os.path.join(ROOT, 'saved_models')

DNN_CFG = {'max_words': 2000}
CNN_CFG = {'vocab_size': 1000, 'embedding_dim': 100, 'n_filters': 64, 'filter_sizes': [2, 3], 'dropout': 0.2, 'batch_size': 32, 'max_len': 200}
BERT_CFG = {'dropout': 0.1, 'batch_size': 8, 'max_len': 128}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')


Device: cuda
  GPU: AMD Radeon Graphics


## 1. Load Data


In [8]:
def load_data(path):
    df = pd.read_csv(path, sep=';')
    df = df[df['Label'].isin(CLASSES)].copy()
    df['label_id'] = df['Label'].map(LABEL_MAP)
    return df


def evaluate_probs(probs, labels, name):
    preds = np.argmax(probs, axis=1)
    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)
    print(f'\n=== {name} ===')
    print(f'Accuracy: {acc:.4f} | Macro F1: {macro_f1:.4f}')
    print(classification_report(labels, preds, target_names=CLASSES, zero_division=0))
    print(pd.DataFrame(confusion_matrix(labels, preds), index=CLASSES, columns=CLASSES))
    return acc, macro_f1


def soft_voting(probabilities, weights=None):
    if weights is None:
        weights = [1.0 / len(probabilities)] * len(probabilities)
    return sum(weight * probs for weight, probs in zip(weights, probabilities))


df_train = load_data(TRAIN_PATH)
df_exemplos = load_data(EXEMPLOS_PATH)
df_full = pd.concat([df_train, df_exemplos], ignore_index=True)
df_es = load_data(EARLY_STOP_PATH)
df_eval = load_data(EVAL_PATH)

print(f'Train final: {len(df_full)} (combined={len(df_train)} + exemplos={len(df_exemplos)})')
print(f'Early stop (subm1): {len(df_es)}')
print(f'Eval (subm2): {len(df_eval)}')


Train final: 4740 (combined=4615 + exemplos=125)
Early stop (subm1): 100
Eval (subm2): 100


## 2. Load DNN Weights


In [9]:
dnn_model = NeuralNetwork(loss=CategoricalCrossEntropy, metric=accuracy, verbose=False)
dnn_model.load(os.path.join(SAVED_MODELS_DIR, 'dnn_final_model.npz'))

with open(os.path.join(SAVED_MODELS_DIR, 'dnn_final_vectorizer.pkl'), 'rb') as f:
    dnn_vectorizer = pickle.load(f)


def predict_dnn(model, vectorizer, texts_raw):
    texts_clean = [clean_text(text) for text in texts_raw]
    X = vectorizer.transform(texts_clean, texts_raw)
    return model.predict(Dataset(X, np.zeros((len(texts_raw), NUM_CLASSES))))


dnn_es = predict_dnn(dnn_model, dnn_vectorizer, df_es['Text'].tolist())
dnn_eval = predict_dnn(dnn_model, dnn_vectorizer, df_eval['Text'].tolist())


## 3. Load CNN1D Weights


In [10]:
cnn_vocab = Vocabulary(max_words=CNN_CFG['vocab_size'])
cnn_vocab.load(os.path.join(SAVED_MODELS_DIR, 'cnn1d_final_vocab.pkl'))

with open(os.path.join(SAVED_MODELS_DIR, 'cnn1d_final_config.json'), 'r') as f:
    cnn_cfg = json.load(f)

style_stats = np.load(os.path.join(SAVED_MODELS_DIR, 'cnn1d_final_style_stats.npz'))
style_mean = style_stats['mean']
style_std = style_stats['std']

with open(os.path.join(SAVED_MODELS_DIR, 'cnn1d_final_style_extractor.pkl'), 'rb') as f:
    cnn_style_extractor = pickle.load(f)

cnn_embeddings = None
if os.path.exists(GLOVE_PATH):
    cnn_embeddings = load_glove_embeddings(GLOVE_PATH, cnn_vocab, embedding_dim=cnn_cfg['embedding_dim'])

cnn_model = CNN1DClassifier(
    vocab_size=len(cnn_vocab),
    embedding_dim=cnn_cfg['embedding_dim'],
    n_filters=cnn_cfg['n_filters'],
    filter_sizes=cnn_cfg['filter_sizes'],
    output_dim=NUM_CLASSES,
    dropout=cnn_cfg['dropout'],
    pretrained_embeddings=cnn_embeddings,
    n_style_features=len(style_mean),
).to(DEVICE)

cnn_model.load_state_dict(torch.load(os.path.join(SAVED_MODELS_DIR, 'cnn1d_final_model.pt'), map_location=DEVICE))
cnn_model.eval()


def predict_cnn(model, vocab, style_extractor, style_mean, style_std, texts_raw):
    texts_clean = [clean_text_sequences(text) for text in texts_raw]
    sequences = texts_to_sequences(texts_clean, vocab, max_len=cnn_cfg['max_len'])
    style = style_extractor.transform(texts_raw)
    style = (style - style_mean) / style_std
    dataset = TextDataset(sequences, np.zeros(len(texts_raw), dtype=np.int64), style)
    loader = DataLoader(dataset, batch_size=CNN_CFG['batch_size'])
    probs = []
    with torch.no_grad():
        for seqs, style_batch, _ in loader:
            seqs = seqs.to(DEVICE)
            style_batch = style_batch.to(DEVICE)
            probs.append(torch.softmax(model(seqs, style_batch), dim=1).cpu().numpy())
    return np.vstack(probs)


cnn_es = predict_cnn(cnn_model, cnn_vocab, cnn_style_extractor, style_mean, style_std, df_es['Text'].tolist())
cnn_eval = predict_cnn(cnn_model, cnn_vocab, cnn_style_extractor, style_mean, style_std, df_eval['Text'].tolist())


GloVe: 1000/1002 palavras encontradas (99.8%)


## 4. Load DistilBERT Weights


In [11]:
bert_tokenizer = get_tokenizer()
bert_model = DistilBERTClassifier(output_dim=NUM_CLASSES, dropout=BERT_CFG['dropout'], freeze_bert=False).to(DEVICE)
bert_model.load_state_dict(torch.load(os.path.join(SAVED_MODELS_DIR, 'distilbert_final_model.pt'), map_location=DEVICE))
bert_model.eval()


def predict_bert(model, texts, labels):
    dataset = DistilBERTDataset(texts, labels, bert_tokenizer, max_len=BERT_CFG['max_len'])
    loader = DataLoader(dataset, batch_size=BERT_CFG['batch_size'])
    probs = []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            logits = model(input_ids, attention_mask)
            probs.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.vstack(probs)


bert_es = predict_bert(bert_model, df_es['Text'].tolist(), df_es['label_id'].values)
bert_eval = predict_bert(bert_model, df_eval['Text'].tolist(), df_eval['label_id'].values)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/home/joaocunha50/studys/university/MEI/1_ano/AP/Projeto-AP/.venv/lib/python3.14/site-packages/transformers/integrations/sdpa_attention.py:92: UserWarning: Mem Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:338.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
/home/joaocunha50/studys/university/MEI/1_ano/AP/Projeto-AP/.venv/lib/python3.14/s

## 5. Individual Results


In [12]:
results = []

dnn_es_acc, dnn_es_f1 = evaluate_probs(dnn_es, df_es['label_id'].values, 'DNN - ES (subm1)')
dnn_eval_acc, dnn_eval_f1 = evaluate_probs(dnn_eval, df_eval['label_id'].values, 'DNN - Eval (subm2)')
results.append({'Model': 'DNN (Forensic)', 'ES_Acc': dnn_es_acc, 'ES_F1': dnn_es_f1, 'Eval_Acc': dnn_eval_acc, 'Eval_F1': dnn_eval_f1})

cnn_es_acc, cnn_es_f1 = evaluate_probs(cnn_es, df_es['label_id'].values, 'CNN1D - ES (subm1)')
cnn_eval_acc, cnn_eval_f1 = evaluate_probs(cnn_eval, df_eval['label_id'].values, 'CNN1D - Eval (subm2)')
results.append({'Model': 'CNN1D (GloVe+Style)', 'ES_Acc': cnn_es_acc, 'ES_F1': cnn_es_f1, 'Eval_Acc': cnn_eval_acc, 'Eval_F1': cnn_eval_f1})

bert_es_acc, bert_es_f1 = evaluate_probs(bert_es, df_es['label_id'].values, 'DistilBERT - ES (subm1)')
bert_eval_acc, bert_eval_f1 = evaluate_probs(bert_eval, df_eval['label_id'].values, 'DistilBERT - Eval (subm2)')
results.append({'Model': 'DistilBERT', 'ES_Acc': bert_es_acc, 'ES_F1': bert_es_f1, 'Eval_Acc': bert_eval_acc, 'Eval_F1': bert_eval_f1})



=== DNN - ES (subm1) ===
Accuracy: 0.6600 | Macro F1: 0.6302
              precision    recall  f1-score   support

   Anthropic       0.82      0.53      0.64        17
      Google       0.56      0.59      0.57        17
       Human       0.77      0.79      0.78        34
        Meta       0.68      0.72      0.70        18
      OpenAI       0.41      0.50      0.45        14

    accuracy                           0.66       100
   macro avg       0.65      0.63      0.63       100
weighted avg       0.68      0.66      0.66       100

           Anthropic  Google  Human  Meta  OpenAI
Anthropic          9       3      2     0       3
Google             1      10      2     1       3
Human              0       2     27     5       0
Meta               0       0      1    13       4
OpenAI             1       3      3     0       7

=== DNN - Eval (subm2) ===
Accuracy: 0.7100 | Macro F1: 0.6858
              precision    recall  f1-score   support

   Anthropic       0.58      0

## 6. Ensemble: Soft Voting


In [15]:
soft_equal_es = soft_voting([dnn_es, cnn_es, bert_es])
soft_equal_eval = soft_voting([dnn_eval, cnn_eval, bert_eval])
se_es_acc, se_es_f1 = evaluate_probs(soft_equal_es, df_es['label_id'].values, 'Soft Vote (equal) - ES')
se_eval_acc, se_eval_f1 = evaluate_probs(soft_equal_eval, df_eval['label_id'].values, 'Soft Vote (equal) - Eval')
results.append({'Model': 'SoftVote (equal)', 'ES_Acc': se_es_acc, 'ES_F1': se_es_f1, 'Eval_Acc': se_eval_acc, 'Eval_F1': se_eval_f1})

soft_bal_es = soft_voting([dnn_es, cnn_es, bert_es], weights=[0.25, 0.25, 0.50])
soft_bal_eval = soft_voting([dnn_eval, cnn_eval, bert_eval], weights=[0.25, 0.25, 0.50])
sb2_es_acc, sb2_es_f1 = evaluate_probs(soft_bal_es, df_es['label_id'].values, 'Soft Vote (0.25/0.25/0.50) - ES')
sb2_eval_acc, sb2_eval_f1 = evaluate_probs(soft_bal_eval, df_eval['label_id'].values, 'Soft Vote (0.25/0.25/0.50) - Eval')
results.append({'Model': 'SoftVote (0.25/0.25/0.50)', 'ES_Acc': sb2_es_acc, 'ES_F1': sb2_es_f1, 'Eval_Acc': sb2_eval_acc, 'Eval_F1': sb2_eval_f1})



=== Soft Vote (equal) - ES ===
Accuracy: 0.7000 | Macro F1: 0.6414
              precision    recall  f1-score   support

   Anthropic       0.73      0.65      0.69        17
      Google       0.67      0.35      0.46        17
       Human       0.74      0.91      0.82        34
        Meta       0.81      0.94      0.87        18
      OpenAI       0.38      0.36      0.37        14

    accuracy                           0.70       100
   macro avg       0.67      0.64      0.64       100
weighted avg       0.69      0.70      0.68       100

           Anthropic  Google  Human  Meta  OpenAI
Anthropic         11       0      3     0       3
Google             3       6      3     0       5
Human              1       1     31     1       0
Meta               0       0      1    17       0
OpenAI             0       2      4     3       5

=== Soft Vote (equal) - Eval ===
Accuracy: 0.7700 | Macro F1: 0.7467
              precision    recall  f1-score   support

   Anthropic      

## 7. Summary


In [ ]:
results_df = pd.DataFrame(results)
results_df

probs_path = os.path.join(ROOT, 'saved_models', 'ensemble_notebook_probs.npz')

np.savez(
    probs_path,
    dnn_es=dnn_es,
    dnn_eval=dnn_eval,
    cnn_es=cnn_es,
    cnn_eval=cnn_eval,
    bert_es=bert_es,
    bert_eval=bert_eval,
    soft_equal_es=soft_equal_es,
    soft_equal_eval=soft_equal_eval,
    soft_bal_es=soft_bal_es,
    soft_bal_eval=soft_bal_eval,
    y_es=df_es['label_id'].values,
    y_eval=df_eval['label_id'].values,
)

print(f'Resultados guardados em {results_path}')
print(f'Probabilidades guardadas em {probs_path}')


Resultados guardados em /home/joaocunha50/studys/university/MEI/1_ano/AP/Projeto-AP/data/processed/ensemble_notebook_results.csv
Probabilidades guardadas em /home/joaocunha50/studys/university/MEI/1_ano/AP/Projeto-AP/saved_models/ensemble_notebook_probs.npz
